# RSNA Knee — LLM prompt comparison (58 true labels)
プロンプトの版(v0/v1/v2)を同じモデル・同じ58件で比べ、確率の平均も試す。`RUN_VARIANT` を設定すると、その版で全レポートにラベルを付けて `labels_out/llm_labels_<版>.csv` に書く。

In [ ]:
import os, sys, subprocess, warnings
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # メモリの断片化によるOOMを減らす
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
REPO = Path('/kaggle/temp/repo') if Path('/kaggle/temp').exists() else Path('/tmp/repo')   # /kaggle/working に置くと実行の出力に含まれてしまう
SRC = REPO/'rsna-knee-abnormality-detection'/'src'
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb', str(REPO)], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
os.environ['HF_HOME'] = '/kaggle/temp/hf' if Path('/kaggle/temp').exists() else '/tmp/hf'   # モデルのキャッシュは出力に含めない
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'accelerate'], check=False)
import time, torch, transformers
import llm_labels as LL
print('transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
from multilingual_labels import label_report_lang
from sklearn.metrics import roc_auc_score
train = pd.read_csv(COMP/'train.csv')
train['has_true'] = train['ACL'].notna()
train = add_pseudo_labels_ml(train)
tv = train[train.has_true].reset_index(drop=True)
T = tv[LABELS].values.astype(int)
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])   # 規則ラベル(正解を見ずに作る)
print('true-labeled:', len(tv), tv.lang.value_counts().to_dict())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
def load(name, four_bit):
    tok = AutoTokenizer.from_pretrained(name)
    kw = dict(device_map='auto')
    if four_bit:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    else:
        kw['torch_dtype'] = torch.float16
    return tok, AutoModelForCausalLM.from_pretrained(name, **kw).eval()
MODEL_NAME, BS = 'Qwen/Qwen2.5-7B-Instruct', 8
try:
    tok, model = load(MODEL_NAME, four_bit=True)
except Exception as e:                       # bitsandbytesが使えない等: 3Bをfp16で
    print('7B 4bit failed ->', repr(e)[:200]); MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    tok, model = load(MODEL_NAME, four_bit=False)
print('model', MODEL_NAME, '| GPU memory GB', round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
VARIANTS = ['v0', 'v1', 'v2']
reports = tv.Report.fillna('').tolist()
res = {}
for v in VARIANTS:
    t0 = time.time()
    H, P, npar, Lv = LL.label_reports(model, tok, reports, bs=BS, variant=v, return_levels=True)
    res[v] = dict(P=P, Lv=Lv, sec=(time.time() - t0) / len(reports), npar=npar)
    print(f'[{v}] {res[v]["sec"]:.2f} s/report -> all {len(train)} reports ~ {res[v]["sec"] * len(train) / 60:.0f} min; '
          f'reports with <12 keys parsed: {int((npar < 12).sum())}')

In [ ]:
def auc(S):
    return np.array([roc_auc_score(T[:, j], S[:, j]) if len(np.unique(T[:, j])) > 1 else np.nan for j in range(len(LABELS))])
scores = {'rule(hard)': R.astype(float), 'v0': res['v0']['P'], 'v2': res['v2']['P'],
          'v1:score(P2+.5*P1)': res['v1']['P'], 'v1:P(>=1)': res['v1']['Lv'][:, :, 1:].sum(-1), 'v1:P(2)': res['v1']['Lv'][:, :, 2]}
scores['v0+v2'] = (scores['v0'] + scores['v2']) / 2
scores['v0+v1score'] = (scores['v0'] + scores['v1:score(P2+.5*P1)']) / 2
scores['v0+v1score+v2'] = (scores['v0'] + scores['v1:score(P2+.5*P1)'] + scores['v2']) / 3
tab = pd.DataFrame({k: auc(S) for k, S in scores.items()}, index=LABELS).round(3)
print(tab.T.assign(macro=tab.mean().round(3)).T if False else tab)
print('macro AUC vs TRUE labels:'); print(tab.mean().round(3).sort_values(ascending=False))

In [ ]:
RUN_VARIANT = None   # 例: 'v2' にすると、その版で全レポートにラベルを付けて labels_out/llm_labels_v2.csv に書く(約4時間)
if RUN_VARIANT:
    out = Path('labels_out'); out.mkdir(exist_ok=True); allr = train.Report.fillna('').tolist(); parts = []
    for s in range(0, len(allr), 400):
        h, p, n = LL.label_reports(model, tok, allr[s:s + 400], bs=BS, variant=RUN_VARIANT, log_every=20)
        parts.append(pd.DataFrame(p, columns=LABELS).assign(parsed=n))
        df = pd.concat(parts, ignore_index=True); df.insert(0, 'StudyInstanceUID', train.StudyInstanceUID.values[:len(df)])
        df.to_csv(out / f'llm_labels_{RUN_VARIANT}.csv', index=False); print('saved', len(df), '/', len(allr))